# Notebook5 - AD Alert Reduction

把多個低階 anomaly flags 聚合為更少、更有語意的 alerts，降低告警疲勞。

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("..").resolve()

DATA_SYNTHETIC = PROJECT_ROOT / "data" / "synthetic"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
flags = pd.read_csv(DATA_PROCESSED / "baseline_anomaly_flags.csv", parse_dates=["timestamp"])
display(flags.head())

## Step 1 - 將 anomaly flags 展開成 raw alerts

In [ ]:
flag_to_metric = {
    "traffic_high": "traffic",
    "packets_high": "packets",
    "errors_high": "errors",
    "discards_high": "discards",
    "broadcast_high": "broadcast",
    "multicast_high": "multicast",
    "unknown_high": "unknown_protocol",
}

raw_rows = []
for _, row in flags.iterrows():
    for flag, metric in flag_to_metric.items():
        if bool(row[flag]):
            raw_rows.append(
                {
                    "timestamp": row["timestamp"],
                    "device_id": row["device_id"],
                    "port_id": row["port_id"],
                    "event_label": row["event_label"],
                    "event_id": row["event_id"],
                    "metric": metric,
                    "alert_name": flag,
                }
            )
raw_alerts = pd.DataFrame(raw_rows)
print(f"raw alerts: {len(raw_alerts):,}")
display(raw_alerts.head(10))

## Step 2 - 視覺化：Raw alert volume

這對應 Grafana 的 raw alert count panel。

In [ ]:
alert_counts = raw_alerts.set_index("timestamp").resample("1h").size()
fig, ax = plt.subplots(figsize=(14, 4))
alert_counts.plot(ax=ax, color="tab:red", linewidth=1)
ax.set_title("Raw alert count per hour")
ax.set_ylabel("raw alerts")
ax.grid(alpha=0.25)
plt.tight_layout()
show_fig(fig)

## Step 3 - 設計問題分類與 15 分鐘聚合規則

In [ ]:
def classify_problem(metrics: set[str], affected_ports: int) -> str:
    if "broadcast" in metrics and affected_ports > 1:
        return "Broadcast storm / L2 loop"
    if "multicast" in metrics and affected_ports > 1:
        return "Multicast flooding"
    if {"traffic", "discards"}.issubset(metrics) and "errors" not in metrics:
        return "Queue congestion"
    if "discards" in metrics:
        return "Queue / buffer pressure"
    if "errors" in metrics and "traffic" not in metrics:
        return "Link quality issue"
    if "unknown_protocol" in metrics:
        return "Unknown protocol / scan"
    if "packets" in metrics and "traffic" in metrics:
        return "Traffic or packet surge"
    if "packets" in metrics:
        return "Packet surge / possible scan"
    if "traffic" in metrics:
        return "Traffic surge / capacity pressure"
    if "broadcast" in metrics:
        return "Broadcast anomaly"
    if "multicast" in metrics:
        return "Multicast anomaly"
    return "General anomaly"


raw_alerts = raw_alerts.sort_values(["port_id", "timestamp"])
raw_alerts["bucket"] = raw_alerts["timestamp"].dt.floor("15min")
reduced_rows = []
for (bucket, alert_name), g in raw_alerts.groupby(["bucket", "alert_name"]):
    metrics = set(g["metric"])
    affected_ports = g["port_id"].nunique()
    severity = min(100, 20 + 8 * len(g) + 15 * affected_ports + 12 * len(metrics))
    reduced_rows.append(
        {
            "start_time": g["timestamp"].min(),
            "end_time": g["timestamp"].max(),
            "problem_type": classify_problem(metrics, affected_ports),
            "affected_ports": ",".join(sorted(g["port_id"].unique())),
            "affected_port_count": affected_ports,
            "evidence_metrics": ",".join(sorted(metrics)),
            "raw_alert_count": len(g),
            "severity_score": severity,
            "event_labels": ",".join(sorted(set(g["event_label"]))),
        }
    )
reduced = pd.DataFrame(reduced_rows).sort_values(["start_time", "severity_score"], ascending=[True, False])
display(reduced.head(12))

## Step 4 - 視覺化：Raw vs Reduced、severity distribution、Top affected ports

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
pd.Series({"raw_alerts": len(raw_alerts), "reduced_alerts": len(reduced)}).plot(kind="bar", ax=axes[0], color=["tab:red", "tab:blue"])
axes[0].set_title("Raw vs Reduced")
axes[0].set_ylabel("count")

reduced["severity_score"].plot(kind="hist", bins=20, ax=axes[1], color="tab:orange")
axes[1].set_title("Severity distribution")

top_ports = raw_alerts["port_id"].value_counts().head(8)
top_ports.sort_values().plot(kind="barh", ax=axes[2], color="tab:green")
axes[2].set_title("Top affected ports")
plt.tight_layout()
show_fig(fig)

## Step 5 - Alert timeline

In [ ]:
timeline = reduced.copy()
timeline["mid_time"] = timeline["start_time"] + (timeline["end_time"] - timeline["start_time"]) / 2
types = {name: i for i, name in enumerate(sorted(timeline["problem_type"].unique()))}
timeline["y"] = timeline["problem_type"].map(types)

fig, ax = plt.subplots(figsize=(14, 6))
scatter = ax.scatter(timeline["mid_time"], timeline["y"], s=timeline["severity_score"] * 2, c=timeline["severity_score"], cmap="viridis", alpha=0.75)
ax.set_yticks(list(types.values()))
ax.set_yticklabels(list(types.keys()))
ax.set_title("Reduced alert timeline")
ax.grid(alpha=0.25)
plt.colorbar(scatter, ax=ax, label="severity")
plt.tight_layout()
show_fig(fig)

In [ ]:
raw_path = DATA_PROCESSED / "raw_alerts.csv"
reduced_path = DATA_PROCESSED / "reduced_alerts.csv"
raw_alerts.to_csv(raw_path, index=False)
reduced.to_csv(reduced_path, index=False)
print(f"saved: {raw_path}")
print(f"saved: {reduced_path}")